Forecasting UK Energy Prices 

In this short project, I aim to use the programming skills I have learnt during my time at university, as well as my own interest in using data to derive insights about the real world problems, to attempt to fit a statistical model to real UK energy data, in hopes to try and make predicitons about the settlement price at the next time period.

Part 1: Retrieving Data about Energy Price:
The data I have used in this project has come from Elexon BMRS: a source of energy data for UK markets. Specifically I collected data starting from the January 1st 2026 to September 23rd 2026 (note only partial data was available on the 23rd September) from the Market Index Price dataset.

In [ ]:
import pathlib
import pandas as pd
import datetime as dt
import time

def generate_data():
    d = dt.date(2026, 1, 1)
    o = dt.date(2026,1,1)
    t=dt.date.today()
    delta = t - o

    index = (delta.days//6) + 1

    for i in range((index)):    
        url = f"https://data.elexon.co.uk/bmrs/api/v1/datasets/MID?from={d}T00%3A00Z&to={d+dt.timedelta(days=5)}T23%3A30Z&dataProviders=APXMIDP&format=csv"
        df = pd.read_csv(url)
        df.to_csv(f"test_{i}.csv", sep=",")
        d += dt.timedelta(days=6)
        time.sleep(0.05)


    df_main = pd.concat([pd.read_csv(f"test_{i}.csv") for i in range((index))]).sort_values(by=["SettlementDate","SettlementPeriod"], ignore_index=True)
    df_main.to_csv("check", sep=",")

    for i in range(index):
        pathlib.Path.unlink(f"test_{i}.csv")
        time.sleep(0.05)

    return df_main

df = generate_data()
df = df.drop(columns=["Unnamed: 0", "DataProvider","Dataset"])
df["StartTime"] = pd.to_datetime(df["StartTime"]).dt.tz_convert(None)
df["SettlementDate"] = pd.to_datetime(df["SettlementDate"])
df.set_index(df["StartTime"], inplace=True)
df = df.drop(columns=["StartTime"])

Before starting to build a model to predict settlement price, I firstly performed EDA an fit a baseline model (Naive Forecast) to (a) understand underlying trends in the data and (b) have some initial metrics to compare my models performance to.

In [ ]:
def mean_sp():    # The settlement period calculations will compute the statistics up to thee latest available settlement period
    mean_settlement_vals = []
    for i in range(1,49):
        mean_settlement_vals.append(df[df["SettlementPeriod"]==i]["Price"].mean())

    return mean_settlement_vals

def sd_sp():
    sd_settlement_vals = []
    for i in range(1,49):
        sd_settlement_vals.append(df[df["SettlementPeriod"]==i]["Price"].std())

    return sd_settlement_vals

def mean_daily():    # The daily calculations will calculate the daily statistics up to the day the program is run
    mean_daily_vals = []
    for i in range(7):
        mean_daily_vals.append(df[df["SettlementDate"].dt.day_of_week==i]["Price"].mean())
        
    return mean_daily_vals

def sd_daily():
    sd_daily_vals = []
    for i in range(7):
        sd_daily_vals.append(df[df["SettlementDate"].dt.day_of_week==i]["Price"].std())
        
    return sd_daily_vals

def mean_monthly():    # The monthly calculations will calculate the monthly statisics up to the day of the month the program is run
    mean_monthly_vals =[]
    for i in range(1,(dt.date.today().month)+1):
        mean_monthly_vals.append(df[df["SettlementDate"].dt.month==i]["Price"].mean())

    return mean_monthly_vals

def sd_monthly():
    sd_monthly_vals =[]
    for i in range(1,(dt.date.today().month)+1):
        sd_monthly_vals.append(df[df["SettlementDate"].dt.month==i]["Price"].std())

    return sd_monthly_vals


# The next function computes the coefficient of variation for each day of the week

def coef_of_var():
    cov_vals = []
    for i in range(7):
        cov_vals.append(df[df["SettlementDate"].dt.day_of_week==i]["Price"].std()/df[df["SettlementDate"].dt.day_of_week==i]["Price"].mean())

    return cov_vals


The below code visualises each of the above situations to give an idea of the shape of the data

In [ ]:
# import matplotlib.pyplot as plt

# prompt = ("Enter the name of the function you want to plot ")
# prompt += "\n(For instance, if you would like to see the mean daily price, enter 'mean_daily()')"

# data_to_collect = input(prompt)

# fig, ax = plt.subplots()

# if 

I'm now going to fit a basseline model. This forecast is a simple naive forecast, where the Price column has has been lagged by 1 settlement period. So the prediciton for the price in the $(n+1)th$ settlement period is just the price in the $nth$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error

series = np.array(df.index.values)
price = np.array(df["Price"].values)


split_time = int(len(df) * 0.7)    # this is the 70th percentile of the dataset: want to use a test size of 0.3

time_train = series[:split_time]
time_test = series[split_time:]

price_train = price[:split_time]
prices_test = price[split_time:]

naive_frcst1 = price[split_time-1: -1]

mse = mean_squared_error(prices_test, naive_frcst1)
rmse = mse ** (1/2)
mae = mean_absolute_error(prices_test, naive_frcst1)

print(rmse, mae)    # These will be the baseline metrics for the models which I will try to fit

Now I have a baseline to work with. Before I build a model for the data, I will frist add some necesary columns, lagging the price and computing the rolling mean price, as this allows the model I will build to capture insight between past and future values.

In [ ]:
df["PriceLag_1"] = df["Price"].shift(1)
df["PriceLag_48"] = df["Price"].shift(48)
df["RollingMean"] = df["Price"].rolling(window=48).mean()
df["RollingSTD"] = df["Price"].rolling(window=48).std()
df["DayOfWeek"] = df.index.day_of_week
df["Month"] = df.index.month

# However now there will be some rows with missing data (cannot compute rolling mean for the first 47 settlement periods etc.)

df = df.dropna()
df

With this information now, we can fit a model. I have chosen to fit a Gradient Boosting Regressor model to this dataset.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor()

feature_cols = ["SettlementPeriod", "DayOfWeek", "Month", "PriceLag_1", "PriceLag_48", "RollingMean", "RollingSTD"]

train_index = int(0.7 * len(df))

X = df[feature_cols]         
y = df['Price']

X_train = X.iloc[:train_index]     
X_test = X.iloc[train_index:]
y_train = y.iloc[:train_index]
y_test = y.iloc[train_index:]

gbr.fit(X_train, y_train)
pred = gbr.predict(X_test)

rmse1 = mean_squared_error(pred, y_test) ** (1/2)
mae1 = mean_absolute_error(pred, y_test)

rmse1, mae1

We can try another model to see if we can imporve the performance and obtain better metrics

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import accuracy_score

model = XGBRegressor(objective='reg:squarederror',
                     n_estimators=100)

model.fit(X_train, y_train)

pred = model.predict(X_test)

mae2 = mean_absolute_error(y_test, pred)
rmse2 = mean_squared_error(y_test, pred)**(1/2)

mae2, rmse2

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rfr = RandomForestRegressor()

rfr.fit(X_train, y_train)

pred2 = rfr.predict(X_test)

mean_absolute_error(y_test, pred), mean_squared_error(y_test, pred)**(1/2)

Visualisation of performance of each model: 